# DROP TARGET DB
#### Duplicate a source DB

In [53]:
import pymongo

def duplicate_db(source, target, uri="mongodb://localhost:27017/"):
    client = pymongo.MongoClient(uri)
    source_db = client[source]
    target_db = client[target]

    # Clear the Database if it Exists
    client.drop_database(target)


    for coll_name in source_db.list_collection_names():
        source_coll = source_db[coll_name]
        target_coll = target_db[coll_name]


        docs = list(source_coll.find({}))
        if docs:
            target_coll.insert_many(docs)
        print(f"Copied {len(docs)} documents from {coll_name}")

    print(f"Database '{source}' successfully duplicated to '{target}'")

duplicate_db("ceur_ws_fix", "ceur_ws_test")

Copied 2267 documents from volumes
Copied 15332 documents from papers
Database 'ceur_ws_fix' successfully duplicated to 'ceur_ws_test'


## Connection

In [64]:
from bson import ObjectId

db = client['ceur_ws_test']


volumes = db['volumes']

papers = db['papers']
related_papers = db['related_papers']
all_papers = db['all_papers']

authors = db['authors']
authors_grouped = db['authors_grouped']

keywords = db['keywords']
keywords_final = db['keywords_final']

abstracts = db['abstracts']

# Create Collections

Related Papers Collection + Associate each Related Paper with an ID and to the Parent Paper.


In [56]:
# Clear the Collection if it Exists
related_papers.delete_many({})


pipeline = [
    {"$unwind": "$paper_info.related_papers"},
    {"$match": {
        "$expr": {
            "$gte": [
                {"$strLenCP": "$paper_info.related_papers.title"},
                20
            ]
        }
    }},
    {"$group": {
        "_id": "$paper_info.related_papers.title",
        "count": {"$sum": 1},
        "paper_ids": {"$push": "$_id"},
        "authors_set": {"$addToSet": "$paper_info.related_papers.authors"},
        "texts": {"$addToSet": "$paper_info.related_papers.text"}
    }},
    {"$sort": {"count": -1}},
    {"$addFields": {
        "cleaned_authors_sets": {
            "$map": {
                "input": "$authors_set",
                "as": "authors",
                "in": {
                    "$filter": {
                        "input": "$$authors",
                        "as": "a",
                        "cond": {
                            "$and": [
                                {"$gt": [{"$strLenCP": {"$trim": {"input": "$$a"}}}, 2]},
                                {"$not": [{"$in": [{"$toLower": "$$a"}, ["et al.", "et al"]]}]},
                                {"$not": [{"$regexMatch": {"input": "$$a", "regex": "BERT", "options": "i"}}]}
                            ]
                        }
                    }
                }
            }
        }
    }},
    {"$addFields": {
        "best_authors": {
            "$reduce": {
                "input": "$cleaned_authors_sets",
                "initialValue": [],
                "in": {
                    "$cond": [
                        {"$gt": [{"$size": "$$this"}, {"$size": "$$value"}]},
                        "$$this",
                        "$$value"
                    ]
                }
            }
        }
    }},
]

results = list(papers.aggregate(pipeline))

for doc in results:
    doc['title'] = doc.pop('_id')
    doc['_id'] = ObjectId()
    doc['text'] = doc['texts'][0] if doc['texts'] else None
    doc.pop('texts', None)
    doc['cleaned_authors_set'] = doc.pop('cleaned_authors_sets', None)
    related_papers.insert_one(doc)

Merge the Related Paper and Papers Collection into an All Papers Collection.

Since the Paper has more Information, the Related Paper is Merged into the Paper Collection.

In [57]:
# Clear the Collection if it Exists
all_papers.delete_many({})
title_to_id = dict()


for paper in papers.find():
    title_to_id[paper['title']] = paper['_id']
    paper['paper_id'] = paper['_id']
    paper['_id'] = ObjectId()
    paper['from'] = 'paper'
    paper['count'] = 1
    all_papers.insert_one(paper)


merged = 0
for rpaper in related_papers.find():
    if rpaper['title'] in title_to_id:
        merged += 1
        all_papers.update_one(
            {'_id': title_to_id[rpaper['title']]},
            {'$inc': {'count': rpaper['count']}}
        )
    else:
        rpaper['paper_id'] = rpaper['_id']
        rpaper['_id'] = ObjectId()
        rpaper['from'] = 'related'
        all_papers.insert_one(rpaper)

print(f"Merged {merged} related papers into existing papers.")

Merged 248 related papers into existing papers.


Authors Collection from the Related Papers and Papers Collections.

In [59]:
# Clear the Collection if it Exists
authors.delete_many({})

# From db["related_papers"]
for doc in related_papers.find():
    for name in doc.get('best_authors', []):
        author_doc = {
            '_id': ObjectId(),
            'name': name,
            'from': 'related',
            'related_id': doc['_id']
        }
        authors.insert_one(author_doc)

# From db["papers"]
for doc in papers.find():
    for name in doc.get('author', []):
        author_doc = {
            '_id': ObjectId(),
            'name': name,
            'from': 'paper',
            'paper_id': doc['_id']
        }
        authors.insert_one(author_doc)


# Clear the Collection if it Exists
authors_grouped.delete_many({})


pipeline = [
    {
        "$group": {
            "_id": "$name",
            "ids": {"$push": "$_id"},
            "from_set": {"$addToSet": "$from"},
            "paper_ids": {"$addToSet": "$paper_id"},
            "related_ids": {"$addToSet": "$related_id"}
        }
    }
]

for doc in authors.aggregate(pipeline):
    author_doc = {
        "_id": ObjectId(),
        "name": doc["_id"],
        "from": list(doc["from_set"]),
        "paper_ids": [pid for pid in doc.get("paper_ids", []) if pid is not None],
        "related_ids": [rid for rid in doc.get("related_ids", []) if rid is not None]
    }
    authors_grouped.insert_one(author_doc)


Keywords Collection from Papers Collection

In [60]:
# Clear the Collection if it Exists
keywords.delete_many({})

# From db["papers"]
for doc in papers.find():
    for kw in doc.get('paper_info', []).get('keywords', []):
        keyword_doc = {
            '_id': ObjectId(),
            'name': kw,
            'paper_id': doc['_id']
        }
        keywords.insert_one(keyword_doc)


# Clear the Collection if it Exists
keywords_final.delete_many({})


pipeline = [
    {
        "$group": {
            "_id": "$name",
            "ids": {"$push": "$_id"},
            "paper_ids": {"$addToSet": "$paper_id"}
        }
    }
]

for doc in keywords.aggregate(pipeline):
    keywords_doc = {
        "_id": ObjectId(),
        "name": doc["_id"],
        "paper_ids": [pid for pid in doc.get("paper_ids", []) if pid is not None]
    }
    keywords_final.insert_one(keywords_doc)

Abstract Collection from Papers Collection.


In [68]:
# Clear the Collection if it Exists
abstracts.delete_many({})

# From db["papers"]
for doc in db['pppp'].find():
    abs_doc = {
        '_id': ObjectId(),
        'text': doc.get('abstract'),
        'paper_id': doc['_id']
    }
    abstracts.insert_one(abs_doc)

# Create Memgraph

### Connection

In [18]:
from gqlalchemy import Memgraph

host_memgraph = "127.0.0.1"
port_memgraph = 7685
memgraph = Memgraph(host=host_memgraph, port=port_memgraph)


# Clear the Database
query_delete = """
    MATCH (n)
    DETACH DELETE n
"""

memgraph.execute(query_delete)

print("Database Cleared.")

def execute_batch(collection, query, batch_size = 10_000):
    batch = []
    batch_counter = 0
    for item in collection:
        item['mongo_id'] = str(item.pop('_id'))

        batch.append(item)

        if len(batch) >= batch_size:
            batch_counter += 1
            memgraph.execute(query, {"batch": batch})
            print(f"Inserted Batch {batch_counter} ({batch_size} Nodes)")

            batch = []

    if batch:
        batch_counter += 1
        memgraph.execute(query, {"batch": batch})
        print(f"Inserted Final Batch {batch_counter} ({len(batch)} items)")

database cleared
Inserted batch 1 (10000 Papers)
Inserted batch 2 (10000 Papers)
Inserted batch 3 (10000 Papers)
Inserted batch 4 (10000 Papers)
Inserted batch 5 (10000 Papers)
Inserted batch 6 (10000 Papers)
Inserted batch 7 (10000 Papers)
Inserted batch 8 (10000 Papers)
Inserted batch 9 (10000 Papers)
Inserted batch 10 (10000 Papers)
Inserted batch 11 (10000 Papers)
Inserted batch 12 (10000 Papers)
Inserted batch 13 (10000 Papers)
Inserted final batch 14 (9031 Papers)


### Nodes

In [ ]:
## ADD Papers

all_papers_select = all_papers.find({}, {
    '_id': 1,
    'title': 1,
    'from': 1,
    'count': 1,
    'text': 1
})

query_add_papers = """
    UNWIND $batch AS row
    CREATE (:Paper {
        id: row.mongo_id,
        name: row.title,
        source: row.from,
        count: row.count,
        text: row.text
    })
"""

execute_batch(all_papers_select, query_add_papers)

memgraph.execute("CREATE INDEX ON :Paper(id)")

In [20]:
## ADD Authors

authors_select = authors.find({}, {
    '_id': 1,
    'name': 1,
    'from': 1
})

query_add_authors = """
    UNWIND $batch as row
    CREATE(:Author {
        id: row.mongo_id,
        name: row.name
        from: row.from
    })
"""

execute_batch(authors_select, query_add_authors)

Inserted batch 1 (10000 Papers)
Inserted batch 2 (10000 Papers)
Inserted batch 3 (10000 Papers)
Inserted batch 4 (10000 Papers)
Inserted batch 5 (10000 Papers)
Inserted batch 6 (10000 Papers)
Inserted batch 7 (10000 Papers)
Inserted batch 8 (10000 Papers)
Inserted batch 9 (10000 Papers)
Inserted batch 10 (10000 Papers)
Inserted batch 11 (10000 Papers)
Inserted batch 12 (10000 Papers)
Inserted batch 13 (10000 Papers)
Inserted batch 14 (10000 Papers)
Inserted batch 15 (10000 Papers)
Inserted batch 16 (10000 Papers)
Inserted batch 17 (10000 Papers)
Inserted batch 18 (10000 Papers)
Inserted batch 19 (10000 Papers)
Inserted batch 20 (10000 Papers)
Inserted batch 21 (10000 Papers)
Inserted batch 22 (10000 Papers)
Inserted batch 23 (10000 Papers)
Inserted batch 24 (10000 Papers)
Inserted batch 25 (10000 Papers)
Inserted batch 26 (10000 Papers)
Inserted batch 27 (10000 Papers)
Inserted batch 28 (10000 Papers)
Inserted batch 29 (10000 Papers)
Inserted batch 30 (10000 Papers)
Inserted batch 31 (

In [ ]:
## ADD Volumes

volumes_select = volumes.find({}, {
    '_id': 1,
    'title': 1,
    'pubyear': 1
})

query_add_volumes = """
    UNWIND $batch as row
    CREATE (:Volume {
        id: row.mongo_id,
        name: row.title,
        year: row.pubyear
    })
"""

execute_batch(volumes_select, query_add_volumes)

In [ ]:
## ADD Keywords

keywords_select = keywords.find({}, {
    '_id': 1,
    'title': 1,

})

query_add_keywords = """
    UNWIND $batch as row
    CREATE (:Keyword {
        id: row.mongo_id,
        name: row.title,
    })
"""

execute_batch(keywords_select, query_add_keywords)

In [ ]:
## ADD Abstracts

abstracts_select = abstracts.find({}, {
    '_id': 1,
    'text': 1,

})

query_add_abstracts = """
    UNWIND $batch as row
    CREATE (:Abstract {
        id: row.mongo_id,
        text: row.text,
    })
"""

execute_batch(abstracts_select, query_add_abstracts)